# BreaQ 2026 — Challenge 2
## The Treacherous Transpiler

**Layer:** circuit compiler and its plugin supply chain  
**Difficulty:** intermediate core; advanced trigger discovery and verification  
**Suggested time:** 4–5 hours, including evidence preparation  
**Prerequisite:** Python and small quantum circuits; Challenge 1 is **not** required

> The report-adapter problem has been contained. BreaQ Cloud now offers a vendor compiler that promises shorter circuits. Calibration jobs look correct, but selected workloads behave differently. Your task is to establish whether the compiler preserves the submitted computation—not merely whether the output circuit runs.

This is an intentionally vulnerable teaching service, **not a discovered flaw in Qiskit or an allegation against a vendor**. All circuits, results, and incidents in this notebook are local simulations. The attack mechanism is original to this lab; research inspiration is identified at the end.

## Mission and team roles

Produce a causal account of a compiler integrity violation, a small reproducer, and a verification gate that detects compromised outputs without rejecting legitimate rewrites.

| Workstream | Suggested owner | First task |
|---|---|---|
| Investigator | Rotate from Challenge 1 | Form hypotheses and compare incident cases |
| Exploit developer | Rotate from Challenge 1 | Discover activating inputs and minimize a witness |
| Defense analyst | Rotate from Challenge 1 | Design tests, characterize false positives, record limitations |

Responsibilities are flexible for one- or two-person teams. After the first 60–90 minutes, exchange findings: each member should explain **another member's result**. Do not leave documentation and presentation to the final few minutes.

**Suggested pacing:** 45 min orientation/triage; 75 min localization; 60 min trigger/reproducer; 60 min defense; 30–60 min extension and submission. Prioritize a defensible core over exhaustive bonus work.

## Threat model and contract

```text
Trusted application / source circuit
                │
                ▼
       compiler service under review
       ├─ normalization
       ├─ phase convention
       ├─ vendor plugin
       └─ native-basis lowering
                │
                ▼
   candidate circuit → trusted local simulator → observations
```

Assume the source circuit, local Qiskit math routines, and local simulator are trusted. The attacker controls a third-party compilation plugin, not the source, verifier, simulator, operating system, or your notebook. Source access is allowed: inspection is a valid investigative path, but it does not replace experiments or generalization.

**Lab contract:** bound measurement-free unitary circuits, 1–6 qubits; no classical bits, dynamic control, ancillas, physical placement, or routing. Input and output logical wires have the same meaning. The compiler promises exact standalone-unitary equivalence **up to global phase**, within numerical tolerance. Basis gates are `rz`, `sx`, `x`, and `cx`. Do not infer production scalability from this deliberately small model.

A global phase is acceptable under this standalone-circuit contract. It may become observable if the whole circuit is later coherently controlled; that use case is outside this contract. Physical-layout-aware comparison is also outside this exercise.

In [ ]:
from pathlib import Path
import csv
import json
import platform
from importlib.metadata import version

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector, state_fidelity

from breaq_ch2 import (
    CompilerService, exact_probabilities, sample_counts,
    total_variation, circuit_digest,
)

service = CompilerService("team_instance.json")
workloads = service.workloads()
print("Team:", service.team_id)
print("Python:", platform.python_version(), "Qiskit:", version("qiskit"))
print("Workloads:", list(workloads))
print(json.dumps(service.contract, indent=2))

## API reference

| API | Result | Use |
|---|---|---|
| `service.workloads()` | Dictionary of circuit copies | Incident inputs |
| `service.compile(qc, build="suspect")` | Candidate circuit | Service under review |
| `service.compile(qc, build="baseline")` | Candidate circuit | Trusted baseline under the stated threat model |
| `service.trace(qc, build="suspect")` | Stage dictionaries with `circuit`, `depth`, `operations` | Local diagnostic snapshots |
| `exact_probabilities(qc)` | Displayed-bit-string → probability | Exact computational-basis distribution from all-zero input |
| `sample_counts(qc, shots=1024, seed=2026)` | Displayed-bit-string → count | Finite-shot ideal local observations |
| `total_variation(p, q)` | Float in `[0, 1]` | Compare **normalized** distributions |
| `circuit_digest(qc)` | SHA-256 of QPY bytes | Artifact identification, **not semantic verification** |

The service never returns an “infected” label. Trace names, metadata, and self-reported statistics are not trust anchors. The local trace API is an educational convenience, not proof that a remote service executed the same pipeline.

## Warm-up: legitimate rewriting

The two circuits below implement the same standalone operation but have different gate sequences. Add a benign phase convention as well. Explain what a verifier should accept.

Qiskit passes operate on a DAG representation; transformations may change syntax without changing meaning. `Operator.equiv` compares operators up to global phase [IBM transpiler documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/transpiler), [IBM Operator API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/qiskit.quantum_info.Operator).

In [ ]:
original = QuantumCircuit(1)
original.h(0)
rewritten = QuantumCircuit(1)
rewritten.rz(np.pi / 2, 0)
rewritten.sx(0)
rewritten.rz(np.pi / 2, 0)
rewritten.global_phase += np.pi / 9
print(original.draw(output="text"))
print(rewritten.draw(output="text"))
print("Equivalent up to global phase:", Operator(original).equiv(Operator(rewritten)))

**Answer.** A verifier should accept any candidate whose unitary equals the source's up to a global phase. Here `h` and `rz(π/2)·sx·rz(π/2)` with an extra global phase π/9 are the same operation, so it must be accepted although the gates, their number and the phase differ. It must refuse a different operator, even one with the same gate count.

## Level 1 — Incident triage

First reproduce sampled observations. Select controls as well as suspicious cases. A distribution from one input and one measurement basis does **not** characterize an entire unitary. Conversely, different finite-shot histograms alone do not prove an integrity violation.

In [ ]:
compiled = {name: service.compile(qc) for name, qc in workloads.items()}
triage_observations = {}
for name in ("CAL-BELL", "SCAN-B", "SCAN-D"):
    triage_observations[name] = {
        "source_counts": sample_counts(workloads[name], shots=512),
        "candidate_counts": sample_counts(compiled[name], shots=512),
    }
    print(name, triage_observations[name])

In [ ]:
# Your experiment: compare more inputs or exact distributions.
# Record the difference between an observation and a security conclusion.
rng = np.random.default_rng(0)

def normalize(counts):
    shots = sum(counts.values())
    return {k: v / shots for k, v in counts.items()}

def noise_floor(probs, shots=512, trials=2000):
    """99th percentile of the TV between two 512-shot samples of the SAME distribution."""
    p = np.array(list(probs.values()))
    a = rng.multinomial(shots, p, size=trials) / shots
    b = rng.multinomial(shots, p, size=trials) / shots
    return float(np.percentile(0.5 * np.abs(a - b).sum(axis=1), 99))

triage_evidence = []
for name, src in workloads.items():
    p_src, p_cand = exact_probabilities(src), exact_probabilities(compiled[name])
    sampled = total_variation(normalize(sample_counts(src, shots=512, seed=1)),
                              normalize(sample_counts(compiled[name], shots=512, seed=2)))  # different seeds!
    triage_evidence.append({"case_id": name, "sampled_tv": round(sampled, 3),
                            "noise_floor_p99": round(noise_floor(p_src), 3),
                            "exact_tv": round(total_variation(p_src, p_cand), 6)})
    print(triage_evidence[-1])

initial_hypotheses = [
    "Observation: only SCAN-B's histogram moves beyond shot noise; CAL-BELL, SCAN-D look identical.",
    "H1: the compiler changes the meaning of some circuits (SCAN-B), not all (CAL-BELL).",
    "H2: identical histograms do not prove equivalence: a phase-only change is invisible in the Z basis.",
    "So a histogram is an observation, not a security conclusion: compare full unitaries next.",
]

### Core investigation

Create a table for **every** workload containing at least:

- Source and compiled gate counts/depth.
- Exact computational-basis total variation distance.
- State fidelity from the all-zero input.
- Full-unitary equivalence up to global phase.

Predict at least one disagreement among those metrics before computing them. Why might a histogram-only or single-input-state checker be insufficient?

In [ ]:
# Implement your own comparison, not a workload-name lookup.
# Prediction before running: SCAN-D keeps the histogram (TV=0) but its unitary differs.
def inspect_compilation(source, candidate):
    return {
        "src_gates": source.size(), "src_depth": source.depth(),
        "cand_gates": candidate.size(), "cand_depth": candidate.depth(),
        "exact_tv": round(total_variation(exact_probabilities(source), exact_probabilities(candidate)), 6),
        "fidelity_000": round(float(state_fidelity(Statevector(source), Statevector(candidate))), 6),
        "unitary_equiv": bool(Operator(source).equiv(Operator(candidate))),  # up to global phase
    }

comparison_rows = []
for name, source in workloads.items():
    comparison_rows.append({"case_id": name, "build": "suspect", **inspect_compilation(source, compiled[name])})
    baseline = service.compile(source, build="baseline")          # control: the trusted build
    comparison_rows.append({"case_id": name, "build": "baseline", **inspect_compilation(source, baseline)})
for row in comparison_rows:
    print(row)
print("Histogram looks fine but unitary differs:",
      [r["case_id"] for r in comparison_rows if r["exact_tv"] == 0 and not r["unitary_equiv"]])

### Threat-model worksheet

Fill this in before looking for implementation details. Which claim is stronger: “same observed output distribution” or “same standalone unitary”? What is actually required by the service contract?

In [ ]:
threat_model = {
    "protected_asset": "the meaning of the program: compiled circuit = source unitary (up to global phase)",
    "attacker_capability": "controls one third-party plugin pass inside the compiler; can rewrite the circuit",
    "trust_boundary": "everything from the compiler service is untrusted (output, trace, stage names, stats)",
    "security_property": "exact standalone-unitary equivalence up to global phase, 1-6 qubits, basis rz/sx/x/cx",
    "trusted_components": ["source circuit", "local Qiskit math (Operator, Statevector)", "local simulator", "our notebook"],
    "out_of_scope": ["large circuits", "controlled use (global phase visible)", "layout/routing/ancillas",
                     "attacker who controls the simulator or our verifier"],
}
print("Stronger claim: 'same standalone unitary' (implies the same histogram for EVERY input and basis).")
print("The contract requires the unitary claim; 'same observed distribution' is weaker.")

## Level 2 — Locate the first violation

Obtain a detached snapshot after each compilation stage. Determine the **first transition** that changes semantics. Do not equate the first syntax change, first phase change, or first reduction in gate count with a violation.

Use source-versus-stage checks or consecutive-stage checks. State whether your conclusion depends on trusting the trace instrumentation.

In [ ]:
focus = "SCAN-B"
trace = service.trace(workloads[focus])
for snapshot in trace:
    print(snapshot["stage"], "depth=", snapshot["depth"], "ops=", snapshot["operations"])

# Available for inspection, but no fault labels are supplied:
# print(trace[index]["circuit"].draw(output="text"))

# Look at the rz sign after the phase stage (02) and after the vendor stage (03).
for s in trace:
    if s["stage"][:2] in ("02", "03"):
        print(s["stage"]); print(s["circuit"].draw(output="text"))

In [ ]:
# Localize the first semantic change. Operator.equiv compares unitaries up to global phase (default atol 1e-8, rtol 1e-5).
def unitary_gap(a, b):
    """Return (equiv, |tr(A^dag B)|/dim, global-phase shift if equivalent)."""
    A, B = Operator(a).data, Operator(b).data
    t = np.trace(A.conj().T @ B) / A.shape[0]
    eq = bool(Operator(a).equiv(Operator(b)))
    return eq, round(float(abs(t)), 6), (round(float(np.angle(t)), 6) + 0.0 if eq else None)

def localize(source, snaps, case_id, build):
    rows, prev = [], source
    for s in snaps:
        eq_prev, ov_prev, shift = unitary_gap(prev, s["circuit"])
        eq_src, ov_src, _ = unitary_gap(source, s["circuit"])
        rows.append({"case_id": case_id, "build": build, "stage": s["stage"], "depth": int(s["depth"]),
                     "equiv_vs_previous": eq_prev, "overlap_vs_previous": ov_prev, "phase_shift_vs_previous": shift,
                     "equiv_vs_source": eq_src, "overlap_vs_source": ov_src})
        prev = s["circuit"]
    return rows

def first_bad(rows):
    return next((i for i, r in enumerate(rows) if not r["equiv_vs_previous"]), None)

main = localize(workloads[focus], trace, focus, "suspect")
control = localize(workloads[focus], service.trace(workloads[focus], build="baseline"), focus, "baseline")
print(f"{'build':9}{'stage':24}{'depth':>5}{'equiv_prev':>12}{'overlap_prev':>14}{'phase_shift':>13}{'equiv_source':>14}")
for r in main + control:
    print(f"{r['build']:9}{r['stage']:24}{r['depth']:>5}{r['equiv_vs_previous']!s:>12}{r['overlap_vs_previous']:>14}"
          f"{r['phase_shift_vs_previous']!s:>13}{r['equiv_vs_source']!s:>14}")

# Controls: every workload on the suspect build; the logic never looks at the names.
allrows = {n: localize(q, service.trace(q), n, "suspect") for n, q in workloads.items()}
verdict = {n: first_bad(rows) for n, rows in allrows.items()}
print("first bad snapshot index per workload:", verdict)
rz = lambda c: [(round(float(g.operation.params[0]), 6), c.find_bit(g.qubits[0]).index) for g in c.data if g.operation.name == "rz"]
print("rz (angle, wire) in each source:", {n: rz(q) for n, q in workloads.items()})
print("gate count source -> compiled:", {n: (len(q.data), len(service.compile(q).data)) for n, q in workloads.items()})
stage_evidence = main + control + [r for n, rows in allrows.items() if n != focus for r in rows]

i = first_bad(main)
assert i is not None, "no bad transition for the focus workload"
before, after = rz(trace[i - 1]["circuit"]), rz(trace[i]["circuit"])
final = service.compile(workloads[focus])
final_ok = Operator(workloads[focus]).equiv(Operator(final))            # independent of the trace
end_ok = Operator(trace[-1]["circuit"]).equiv(Operator(final))          # last snapshot agrees with the real output
phase_only = {r["stage"]: r["phase_shift_vs_previous"] for r in main if r["equiv_vs_previous"] and abs(r["phase_shift_vs_previous"]) > 1e-9}
print(f"rz (angle, wire) before {before} after {after}; final output equivalent to source: {final_ok}; last snapshot matches compile(): {end_ok}")
first_bad_transition = (f"{trace[i-1]['stage']} -> {trace[i]['stage']}: unitary changes (overlap {main[i]['overlap_vs_previous']}); "
                        f"rz (angle, wire) {before} -> {after}. Phase-only stages (allowed): {phase_only}. "
                        f"Localization trusts the trace snapshots (untrusted service output). Independent anchors: snapshot 00 equals our source "
                        f"({main[0]['equiv_vs_source']}), the last snapshot matches service.compile ({end_ok}), and the compiled circuit is "
                        f"{'equivalent' if final_ok else 'NOT equivalent'} to the source.")
print(first_bad_transition)

### Trigger discovery

Form and test a hypothesis about **when** the service changes semantics. Change one feature at a time: parameters, gate positions, qubit roles, spectators, and input-state preparation. Try both predicted trigger and predicted non-trigger cases.

Source inspection is permitted. If used, cite the relevant logic and then verify the inferred condition experimentally. The notebook must stand on evidence even when a jury instance uses different numeric parameters.

In [ ]:
# Black-box trigger hunt: own circuits -> service.compile -> Operator.equiv vs source. One feature per row.
# Source reading (breaq_ch2/service.py: _EchoBalance.run lines 75-89, _profile lines 52-56) only gives predictions; each row tests one.
# Operator.equiv uses atol 1e-8 / rtol 1e-5 by default; the flips below are >= 0.2 rad, far above that.
import tempfile, time
t0 = time.time()

def circ(*ops, n=3):
    qc = QuantumCircuit(n)
    for name, *args in ops: getattr(qc, name)(*args)
    return qc

def flips(svc, qc, control=False):
    """True if the suspect output breaks equivalence (and, with control=True, the baseline does not)."""
    bad = not Operator(qc).equiv(Operator(svc.compile(qc, build="suspect")))
    return bad and (not control or Operator(qc).equiv(Operator(svc.compile(qc, build="baseline"))))

def discover(svc, a=2 * np.pi, step=0.01):
    """Grid-sweep the angle of cx(0,1) rz(angle,1) cx(0,1), then bisect both edges to 1e-6."""
    grid = np.arange(-a, a, step)
    hits = np.array([flips(svc, circ(("cx", 0, 1), ("rz", g, 1), ("cx", 0, 1))) for g in grid])
    idx = np.flatnonzero(hits)
    assert len(idx) and np.all(np.diff(idx) == 1), "no hit, or hits are not one interval"
    def bisect(out, inn):
        while abs(inn - out) > 1e-6:
            m = (inn + out) / 2
            out, inn = (out, m) if flips(svc, circ(("cx", 0, 1), ("rz", m, 1), ("cx", 0, 1))) else (m, inn)
        return inn
    return bisect(grid[idx[0] - 1], grid[idx[0]]), bisect(grid[idx[-1] + 1], grid[idx[-1]]), step

low, high, step = discover(service)
m = (low + high) / 2
print(f"activation interval ~ [{low:.6f}, {high:.6f}] rad, width {high-low:.6f}; grid step {step}, edges bisected to 1e-6")

P = lambda *mid, n=3: circ(("h", 0), ("cx", 0, 1), *mid, ("cx", 0, 1), n=n)
features = [  # (feature varied, circuit, predicted trigger)
    ("reference: h(0) cx(0,1) rz(mid,1) cx(0,1)", P(("rz", m, 1)), True),
    ("angle 1e-3 below lower edge", P(("rz", low - 1e-3, 1)), False),
    ("angle 1e-3 above upper edge", P(("rz", high + 1e-3, 1)), False),
    ("angle 1e-3 inside lower edge", P(("rz", low + 1e-3, 1)), True),
    ("angle 1e-3 inside upper edge", P(("rz", high - 1e-3, 1)), True),
    ("negative angle -mid", P(("rz", -m, 1)), False),
    ("angle mid + 2*pi (same rotation, other number)", P(("rz", m + 2 * np.pi, 1)), False),
    ("rz on the control wire", P(("rz", m, 0)), False),
    ("cx reversed: rz on qubit 1 (now control)", circ(("cx", 1, 0), ("rz", m, 1), ("cx", 1, 0)), False),
    ("cx reversed: rz on qubit 0 (now target)", circ(("cx", 1, 0), ("rz", m, 0), ("cx", 1, 0)), True),
    ("closing cx has other orientation", circ(("cx", 0, 1), ("rz", m, 1), ("cx", 1, 0)), False),
    ("cz instead of cx", circ(("cz", 0, 1), ("rz", m, 1), ("cz", 0, 1)), False),
    ("x on target between cx and rz", P(("x", 1), ("rz", m, 1)), False),
    ("x on control between (target wire untouched)", P(("x", 0), ("rz", m, 1)), True),
    ("spectator x on third qubit inside the pattern", P(("x", 2), ("rz", m, 1)), True),
    ("barrier inside the pattern", P(("barrier",), ("rz", m, 1)), True),
    ("no input prep (no h)", circ(("cx", 0, 1), ("rz", m, 1), ("cx", 0, 1)), True),
    ("control starts in |1> (x), no h", circ(("x", 0), ("cx", 0, 1), ("rz", m, 1), ("cx", 0, 1)), True),
    ("both qubits in superposition (h, h)", circ(("h", 0), ("h", 1), ("cx", 0, 1), ("rz", m, 1), ("cx", 0, 1)), True),
    ("rz split in two halves", P(("rz", m / 2, 1), ("rz", m / 2, 1)), False),
    ("p(mid) instead of rz(mid)", P(("p", m, 1)), False),
]
trigger_experiments = []
def record(svc, feature, qc, predicted, observed=None):
    observed, predicted = bool(flips(svc, qc, control=True) if observed is None else observed), bool(predicted)
    trigger_experiments.append({"feature_varied": feature, "instance_seed": svc.seed, "predicted_trigger": predicted,
                                "observed_trigger": observed, "match": predicted == observed})
for f, qc, pred in features: record(service, f, qc, pred)

# Several patterns (spec = [(target, angle)]): which rz is negated? Compare the output with "only rz #k negated".
def pats(spec, flip=()):
    qc = circ(("h", 0))
    for k, (t, a) in enumerate(spec):
        qc.cx(0, t); qc.rz(-a if k in flip else a, t); qc.cx(0, t)
    return qc
def negated(svc, spec):
    out = Operator(svc.compile(pats(spec)))
    return [k for k in range(len(spec)) if out.equiv(Operator(pats(spec, (k,))))]
for feature, spec, expect in [
        ("two patterns, targets 1 then 2: only the first rz is negated", [(1, m), (2, m)], [0]),
        ("two patterns, target 2 written first: the lower wire (1) is negated, not the earlier gate", [(2, m), (1, m)], [1]),
        ("a pattern outside the window does not block a later one", [(1, low - 0.1), (2, m)], [1])]:
    record(service, feature, None, True, negated(service, spec) == expect)

# Generalization: a second instance (other seed, temp file) must show the same behaviour with a shifted interval.
rule = lambda s: (0.25 + 0.025 * (s % 7), 0.75 + 0.025 * (s % 7))     # copied from _profile (service.py lines 54-55), a prediction only; a False row means a different build
edges_ok = lambda svc, lo, hi: max(abs(lo - rule(svc.seed)[0]), abs(hi - rule(svc.seed)[1])) < 1e-5
record(service, "measured edges equal the source rule", None, True, edges_ok(service, low, high))
with tempfile.TemporaryDirectory() as d:
    cfg = Path(d) / "other.json"
    cfg.write_text(json.dumps({"schema": "breaq-ch2-v1", "team_id": "jury-test", "seed": service.seed + 3}))
    other = CompilerService(str(cfg))
low2, high2, _ = discover(other)
print(f"seed {other.seed}: interval ~ [{low2:.6f}, {high2:.6f}], width {high2-low2:.6f}  (shift of lower edge {low2-low:+.6f})")
record(other, "measured edges equal the source rule", None, True, edges_ok(other, low2, high2))
for name, a in [("old lower edge + 0.01", low + 0.01), ("old upper edge - 0.01", high - 0.01), ("new interval centre", (low2 + high2) / 2)]:
    record(other, f"second instance: angle at {name}", P(("rz", a, 1)), low2 <= a <= high2)

for r in trigger_experiments: print(r["match"], "|", r["instance_seed"], "|", r["feature_varied"], "| pred", r["predicted_trigger"], "obs", r["observed_trigger"])
print("all predictions matched:", all(r["match"] for r in trigger_experiments), f"({len(trigger_experiments)} rows, {time.time()-t0:.1f}s)")
trigger_hypothesis = (f"The suspect build negates an rz angle when (1) the angle lies in [{low:.6f}, {high:.6f}] rad "
    f"(grid step {step}, edges bisected to 1e-6; the window shifts with the instance seed: {low2:.6f}..{high2:.6f} for seed {other.seed}), "
    "(2) the rz sits on the TARGET wire of a cx pair with identical qubits and nothing else on that wire between them. "
    "(3) At most one rz is negated per compile: the matching pattern on the lowest-numbered target wire, whatever the gate order "
    "in the circuit; a pattern outside the window does not block a later one. "
    "Not triggered by: rz on the control, a gate on the target wire between rz and cx, p or split-rz syntax, a different angle number "
    "(negative, +2pi). Spectators, barriers and input preparation are irrelevant. Matches _EchoBalance in service.py.")
print(trigger_hypothesis)

### Hint ladder — optional

Hints are visible for practice. For event-time controlled release, the organizer can remove this cell and release the same text separately. Self-reported hint use is not an enforceable scoring mechanism.

<details><summary>Hint 1 — Comparing meanings</summary>
A valid basis translation can replace an entire gate sequence. Compare the promised operation, not its spelling or its length.
</details>

<details><summary>Hint 2 — Observability</summary>
Relative phases can matter even when a computational-basis histogram is unchanged. Test a different input or add a trusted analysis basis **after** compilation.
</details>

<details><summary>Hint 3 — Fault localization</summary>
Compare unitary operators at adjacent trace stages; skip global-phase differences allowed by the contract.
</details>

<details><summary>Hint 4 — Counterfactual experiment</summary>
Compare the two compiler builds on related parameterized circuits. Track whether changing a rotation's location or direction changes activation.
</details>

## Minimal reproducer and observable witness

Produce two related artifacts:

1. A small source circuit for which the suspect compiler violates the equivalence contract.
2. A trusted preparation and analysis experiment that makes the difference observable.

A tiny reproducer need not show a difference on all-zero input. The preparation and analysis may be outside the submitted subroutine; make that distinction explicit. Record your minimization criterion and which deletions you tried. Renaming a supplied workload is not minimization.

In [ ]:
# Minimal reproducer + observable witness. Run the trigger cell first: this cell uses its low, high (measured interval).
import itertools
def build(ops, n):
    qc = QuantumCircuit(n)
    for op, q in ops: qc.append(op, q)
    return qc
def text(c): return [f"{i.operation.name}{'(%.3f)' % i.operation.params[0] if i.operation.params else ''} q{[c.find_bit(q).index for q in i.qubits]}" for i in c.data]
def prop(qc):
    """(suspect output NOT equivalent to source, baseline output equivalent to source)."""
    return (not Operator(qc).equiv(Operator(service.compile(qc, build="suspect"))),
            bool(Operator(qc).equiv(Operator(service.compile(qc, build="baseline")))))

# 1. Greedy minimization from SCAN-B: delete one instruction, keep the deletion only if the property still holds.
minimization_criterion = "keep a deletion only if the suspect output is NOT equivalent to the source AND the baseline output IS (Operator.equiv, up to global phase)"
src = workloads["SCAN-B"]
ops = [(i.operation, tuple(src.find_bit(q).index for q in i.qubits)) for i in src.data]
deletions_tried, n_pass, changed = [], 0, True
while changed:
    n_pass, changed, i = n_pass + 1, False, 0
    while i < len(ops):
        trial = ops[:i] + ops[i + 1:]
        bad, good = prop(build(trial, src.num_qubits))
        name = text(build([ops[i]], src.num_qubits))[0]
        deletions_tried.append({"pass": n_pass, "index": i, "deleted": name, "suspect_equiv": not bad, "baseline_equiv": good, "kept": bad and good})
        print(f"pass {n_pass} #{i}: delete {name:16} suspect_equiv={not bad!s:5} baseline_equiv={good!s:5} -> {'KEPT' if bad and good else 'rejected'}")
        if bad and good: ops, changed = trial, True
        else: i += 1

# 2. Rebuild on the fewest qubits, theta = middle of the measured interval, and check it is 1-minimal.
used = sorted({q for _, qs in ops for q in qs})
ops = [(op, tuple(used.index(q) for q in qs)) for op, qs in ops]
assert [(o.name, q) for o, q in ops] == [("cx", (0, 1)), ("rz", (1,)), ("cx", (0, 1))], "unexpected survivor"
theta = (low + high) / 2
minimal_source = QuantumCircuit(len(used)); minimal_source.cx(0, 1); minimal_source.rz(theta, 1); minimal_source.cx(0, 1)
bad, good = prop(minimal_source)
assert bad and good and not any(all(prop(build(ops[:k] + ops[k + 1:], 2))) for k in range(3))
lib = [(g, (q,)) for g in ("h", "x", "sx", "s", "rz") for q in (0, 1)] + [("cx", (0, 1)), ("cx", (1, 0))]
def mk(seq):
    qc = QuantumCircuit(2)
    for g, q in seq: qc.rz(theta, *q) if g == "rz" else getattr(qc, g)(*q)
    return qc
smaller = [s for k in (1, 2) for s in itertools.product(lib, repeat=k) if all(prop(mk(s)))]
n_smaller = sum(len(lib) ** k for k in (1, 2))
print("minimal_source:", text(minimal_source), f"theta={theta:.6f} suspect_equiv={not bad} baseline_equiv={good}")
print(f"all 3 single deletions fail the property; {len(smaller)} of {n_smaller} circuits with 1-2 instructions (12-gate library) satisfy it")

# 3. Witness: the TESTER adds prep and analysis AFTER compilation (the compiler never sees them).
prep = QuantumCircuit(2); prep.h(0); prep.s(0)       # control -> |+i>, target stays |0>
ana = QuantumCircuit(2); ana.h(0)                     # turns the control's relative phase into a population
witness = lambda c: prep.compose(c).compose(ana)
cand = {b: service.compile(minimal_source, build=b) for b in ("baseline", "suspect")}
P0 = lambda p: sum(v for k, v in p.items() if k[-1] == "0")   # P(q0 = 0); last character of the string is q0
ideal = exact_probabilities(witness(minimal_source))
exact = {b: exact_probabilities(witness(c)) for b, c in cand.items()}
shots, shot_seed = 4096, 5                                       # sampling seed: free choice, not the instance seed
counts = {b: sample_counts(witness(c), shots=shots, seed=shot_seed) for b, c in cand.items()}
norm = lambda d: {k: v / sum(d.values()) for k, v in d.items()}
zero_tv = total_variation(exact_probabilities(cand["baseline"]), exact_probabilities(cand["suspect"]))
tv_exact, tv_sampled = total_variation(exact["baseline"], exact["suspect"]), total_variation(*map(norm, counts.values()))
assert zero_tv < 1e-9, "the bare circuit should be invisible from |00>"
assert np.isclose(tv_exact, abs(np.sin(theta)), atol=1e-9) and np.isclose(P0(exact["baseline"]), (1 - np.sin(theta)) / 2)
print(f"bare minimal_source from |00>: TV(baseline, suspect) = {zero_tv:.1e}  (no difference)")
print(f"witness P(q0=0): source {P0(ideal):.4f}, baseline {P0(exact['baseline']):.4f}, suspect {P0(exact['suspect']):.4f}; formula (1-sin t)/2 = {(1-np.sin(theta))/2:.4f}")
print(f"witness exact TV = {tv_exact:.6f}, |sin theta| = {abs(np.sin(theta)):.6f}; sampled TV ({shots} shots) = {tv_sampled:.4f}")
print("counts baseline:", counts["baseline"], "| suspect:", counts["suspect"])

reproducer_evidence = {
    "theta": float(theta), "interval_used": [float(low), float(high)], "minimal_source_instructions": text(minimal_source),
    "suspect_equiv": not bad, "baseline_equiv": good, "minimization_start": f"SCAN-B workload ({len(src.data)} instructions, {src.num_qubits} qubits)",
    "minimization_criterion": minimization_criterion, "deletions_tried": deletions_tried,
    "smaller_circuits_satisfying": len(smaller), "smaller_circuits_checked": n_smaller,
    "witness": {"prep": "h(q0); s(q0): control in |+i>, target |0>", "analysis": "h(q0), then Z-measure",
                "applied": "tester composes prep + compiled candidate + analysis AFTER compilation, outside the submitted subroutine"},
    "zero_input_tv": float(zero_tv), "witness_exact_tv": float(tv_exact), "abs_sin_theta": float(abs(np.sin(theta))),
    "witness_p_q0_zero": {"source": float(P0(ideal)), "baseline": float(P0(exact["baseline"])), "suspect": float(P0(exact["suspect"]))},
    "witness_sampled_tv": float(tv_sampled), "shots": shots, "sampling_seed": shot_seed, "witness_counts": counts,
    "instance_seed": service.seed, "label": "local simulation"}
assert json.dumps(reproducer_evidence, allow_nan=False)

## Level 3 — Verification gate

Implement `verify_compilation(source, candidate) -> bool`:

- Return `True` only for equivalent in-contract circuits, up to global phase.
- Return `False` for a semantic mismatch, unsupported input, incompatible width, or failed validation.
- Do not trust circuit names, workload IDs, metadata, counts, stage labels, or byte hashes.
- Respect the 6-qubit bound; do not claim a full-matrix method scales to useful large circuits.

Then implement a wrapper that obtains a candidate, verifies it independently, and either returns it or **refuses execution**. Removing one known-bad gate or checking a specific trigger is not a general verifier.

This is small-system numerical translation validation, not a formal proof or a scalable defense for arbitrary quantum programs.

In [ ]:
# Independent check: compare the SOURCE and the CANDIDATE as matrices. Fail closed on anything odd.
import math
from numbers import Real

def verify_compilation(source, candidate) -> bool:
    """True only if candidate = source up to a global phase (spectral-norm distance <= 1e-8) and both are
    plain bound circuits of equal width 1..6; the candidate may use only rz/sx/x/cx (+ barrier, id).
    Numerical check of a 2^n x 2^n matrix, not a proof. Never reads circuit names, metadata, counts or hashes."""
    lab = {"id", "h", "x", "y", "z", "s", "sdg", "t", "tdg", "sx", "sxdg", "rx", "ry", "rz",
           "p", "cx", "cz", "swap", "barrier"}                    # the lab's own allowed gates
    basis = {"rz", "sx", "x", "cx", "id", "barrier"}              # policy: id/barrier = no unitary effect, lab allows them

    def ok(qc, gates, max_ops):                                    # structure checks, no matrices yet
        if not isinstance(qc, QuantumCircuit) or not 1 <= qc.num_qubits <= 6:
            return False
        if qc.num_clbits or qc.num_ancillas or qc.parameters or qc.layout is not None:
            return False
        if len(qc.data) > max_ops or not math.isfinite(float(qc.global_phase)):
            return False
        return all(i.operation.name in gates and not i.clbits and
                   all(isinstance(p, Real) and math.isfinite(float(p)) for p in i.operation.params)
                   for i in qc.data)

    def plain(qc):                                                 # re-make every gate from (name, angles, wires) with Qiskit's own code
        out = QuantumCircuit(qc.num_qubits, global_phase=qc.global_phase)
        for i in qc.data:
            getattr(out, i.operation.name)(*i.operation.params, *[qc.find_bit(q).index for q in i.qubits])
        return out
    try:
        if not (ok(source, lab, 500) and ok(candidate, basis, 5000)):
            return False
        if source.num_qubits != candidate.num_qubits:
            return False
        u, v = Operator(source).data, Operator(candidate).data
        for qc, m in ((source, u), (candidate, v)):                # a fake gate object whose own matrix disagrees with its name+angle
            if not np.allclose(Operator(plain(qc)).data, m, atol=1e-12):
                return False
        z = np.vdot(u, v)                                          # tr(U^dagger V) gives the best global phase
        if abs(z) < len(u) / 2:
            return False                                           # far from any phase multiple of U
        return bool(np.linalg.norm(v - z / abs(z) * u, 2) <= 1e-8)  # tolerance: spectral norm <= 1e-8
    except Exception:
        return False                                               # fail closed


def guarded_compile(compiler, source, build="suspect"):
    try:
        pristine = source.copy()                                   # we verify against our own copy of the source,
        candidate = compiler.compile(source.copy(), build=build).copy()   # never the objects the compiler could edit
    except Exception as exc:
        raise ValueError("Compilation refused: compiler failed") from exc
    if not verify_compilation(pristine, candidate):
        raise ValueError("Compilation refused: integrity not established")
    return candidate

In [ ]:
import random, inspect
import importlib.util
from collections import Counter
from qiskit import transpile
from qiskit.circuit import AncillaRegister, Gate, Parameter, QuantumRegister
from qiskit.circuit.library import RZGate

trng = random.Random(7)                                    # test-suite seed: reproducible, unrelated to the instance seed
ONE = ["id", "h", "x", "y", "z", "s", "sdg", "t", "tdg", "sx", "sxdg"]
BASIS = service.contract["basis_gates"]

# ---- 1. Case builders: random lab circuits, and one-edit mutations of a legitimate candidate ----
def rand_circuit(n):                                       # random lab circuit, 4..14 steps
    qc = QuantumCircuit(n)
    for _ in range(trng.randint(4, 14)):
        r = trng.random()
        if n > 1 and r < 0.15:                             # cx-rz-cx motif (shape seen in the plugin code), random angle 0..1.2
            a, b = trng.sample(range(n), 2); qc.cx(a, b); qc.rz(trng.uniform(0, 1.2), b); qc.cx(a, b)
        elif n > 1 and r < 0.35:
            getattr(qc, trng.choice(["cx", "cz", "swap"]))(*trng.sample(range(n), 2))
        elif r < 0.5:
            getattr(qc, trng.choice(["rx", "ry", "rz", "p"]))(trng.uniform(-6.3, 6.3), trng.randrange(n))
        elif r < 0.55:
            qc.barrier()
        else:
            getattr(qc, trng.choice(ONE))(trng.randrange(n))
    return qc

def mutate(qc, kind):                                      # change ONE thing in a legitimate candidate
    pick = {"flip-rz-sign": "rz", "shift-angle-1e-3": "rz", "reverse-cx": "cx"}.get(kind)
    pos = [j for j, ins in enumerate(qc.data) if pick is None or ins.operation.name == pick]
    if not pos:
        return None
    j, out = trng.choice(pos), QuantumCircuit(qc.num_qubits, global_phase=qc.global_phase)
    for k, ins in enumerate(qc.data):
        op, q = ins.operation, ins.qubits
        if k == j and kind == "drop-gate": continue
        if k == j and kind == "flip-rz-sign": op = RZGate(-op.params[0])
        if k == j and kind == "shift-angle-1e-3": op = RZGate(op.params[0] + 1e-3)
        if k == j and kind == "reverse-cx": q = q[::-1]
        out.append(op, q)
    if kind == "add-x": out.x(trng.randrange(qc.num_qubits))
    return out

# ---- 2. Independent label: 8 random states, overlaps must all have modulus 1 and one common phase ----
def oracle_equal(src, cand, k=8):                          # resolves errors of about 1e-4 rad or more
    g, ov = np.random.default_rng(5), []
    for _ in range(k):
        v = g.normal(size=2 ** src.num_qubits) + 1j * g.normal(size=2 ** src.num_qubits)
        psi = Statevector(v / np.linalg.norm(v))
        ov.append(np.vdot(psi.evolve(src).data, psi.evolve(cand).data))
    return abs(np.mean(ov)) > 1 - 1e-10

# ---- 3. The case list (kind, source, candidate) and the invalid inputs ----
cases = []
for _ in range(20):
    src = rand_circuit(trng.randint(1, 4)); base = service.compile(src, build="baseline")
    ph = base.copy(); ph.global_phase += trng.uniform(-3, 3)
    cases += [("baseline", src, base), ("baseline+phase", src, ph), ("suspect-random", src, service.compile(src, build="suspect"))]
    cases += [(f"qiskit-opt{o}", src, transpile(src, basis_gates=BASIS, optimization_level=o, seed_transpiler=1)) for o in (0, 1)]
    for kind in ("flip-rz-sign", "shift-angle-1e-3", "drop-gate", "add-x", "reverse-cx"):
        if (m := mutate(base, kind)) is not None: cases.append(("mut:" + kind, src, m))
for s, c in zip(workloads.values(), compiled.values()):   # labels come from the oracle, never from names
    cases += [("suspect-workload", s, c), ("baseline-workload", s, service.compile(s, build="baseline"))]

class Fake(Gate):                                          # says "x" but its matrix is the identity
    def __init__(self): super().__init__("x", 1, [])
    def __array__(self, dtype=None, copy=None): return np.eye(2, dtype=complex)
one, two, big, hh, toff, long = (QuantumCircuit(n) for n in (1, 2, 7, 1, 3, 1))
one.x(0); two.x(0); two.x(1); big.x(0); hh.h(0); toff.ccx(0, 1, 2)
for _ in range(501): long.x(0)
meas = one.copy(); meas.measure_all()
par = QuantumCircuit(1); par.rz(Parameter("t"), 0)
nan = QuantumCircuit(1); nan.rz(float("nan"), 0)
fake = QuantumCircuit(1); fake.append(Fake(), [0])
lay = transpile(two, basis_gates=BASIS, coupling_map=[[0, 1], [1, 0]], initial_layout=[1, 0], seed_transpiler=1)
anc = QuantumCircuit(QuantumRegister(1), AncillaRegister(1)); anc.x(0); anc.x(1)
bad = [("width-mismatch", one, two), ("7-qubits", big, big.copy()), ("measurement", one, meas), ("clbits", one, QuantumCircuit(1, 1)),
       ("parameters", par, par.copy()), ("nan-angle", nan, nan.copy()), ("h-in-candidate", hh, hh.copy()),
       ("source-gate-outside-lab", toff, toff.copy()), ("501-instructions", long, long.copy()), ("not-a-circuit", one, None),
       ("string-source", "x", one.copy()), ("numpy-candidate", one, np.eye(2)), ("fake-x-gate", QuantumCircuit(1), fake),
       ("layout", two, lay), ("ancilla", two, anc)]

# ---- 4. Score: positive class = NOT equivalent ----
pos = [not oracle_equal(s, c) for _, s, c in cases]
refused = [not verify_compilation(s, c) for _, s, c in cases]
hits = Counter(zip(pos, refused))                          # (is it really different?, did we refuse it?)
TP, FN, TN, FP = hits[True, True], hits[True, False], hits[False, False], hits[False, True]
refused_invalid = sum(not verify_compilation(s, c) for _, s, c in bad)
by_kind = {k: {"n": n, "positives": sum(p for (kk, _, _), p in zip(cases, pos) if kk == k),
               "refused": sum(r for (kk, _, _), r in zip(cases, refused) if kk == k)} for k, n in Counter(k for k, _, _ in cases).items()}
print(f"cases={len(cases)} positives={TP + FN} negatives={TN + FP}  TP={TP} FN={FN} TN={TN} FP={FP}  invalid refused {refused_invalid}/{len(bad)}")
for k, v in by_kind.items(): print(f"  {k:20s}", v)
print("mutations the oracle says changed nothing:", [kk for (kk, _, _), p in zip(cases, pos) if kk.startswith("mut:") and not p])
agree = [(p == (not Operator(service.compile(s, build="baseline")).equiv(c))) for (kk, s, c), p in zip(cases, pos) if kk.startswith("suspect")]
print(f"oracle vs 'differs from trusted baseline' on suspect outputs: {sum(agree)}/{len(agree)} agree")
o0 = [r for (kk, _, c), r in zip(cases, refused) if kk == "qiskit-opt0" and "barrier" in c.count_ops()]
print(f"qiskit-opt0 outputs that carry a barrier: {len(o0)}, refused among them: {sum(o0)}")
o2 = [(s, transpile(s, basis_gates=BASIS, optimization_level=2, seed_transpiler=1)) for k, s, _ in cases if k == "baseline"]
lay2 = [(s, c) for s, c in o2 if c.layout is not None]     # Qiskit level 2 may elide a swap and record it as a layout
print(f"qiskit-opt2 outputs with a layout: {len(lay2)} of {len(o2)}; correct when layout-aware: "
      f"{all(Operator.from_circuit(c).equiv(Operator(s)) for s, c in lay2)}; refused by us (out of contract): {all(not verify_compilation(s, c) for s, c in lay2)}")

# ---- 5. Tolerance, the wrapper, and a hostile compiler ----
demo = QuantumCircuit(2); demo.sx(0); demo.cx(0, 1); demo.rz(0.7, 1)
def nudge(d): c = QuantumCircuit(2); c.sx(0); c.cx(0, 1); c.rz(0.7 + d, 1); return c
tolerance_demo = {f"{d:g}": verify_compilation(demo, nudge(d)) for d in (1e-10, 1e-9, 1e-8, 3e-8, 1e-7, 1e-3)}
print("angle error -> accepted?", tolerance_demo)

guard = {b: {"returned": [], "refused": []} for b in ("suspect", "baseline")}   # names are display text only
for name, s in workloads.items():
    for build in guard:
        try: guarded_compile(service, s, build=build); guard[build]["returned"].append(name)
        except ValueError: guard[build]["refused"].append(name)
for b, g in guard.items(): print(f"guarded_compile {b:8s} refused {g['refused']}  returned {g['returned']}")
tv = {n: round(total_variation(exact_probabilities(workloads[n]), exact_probabilities(compiled[n])), 3) for n in guard["suspect"]["refused"]}
print("Z-histogram TV of the refused outputs (0 = a histogram check would pass them):", tv)

class Hostile:                                             # compiler that also rewrites the caller's circuit to match its output
    def compile(self, qc, build="suspect"):
        out = service.compile(qc, build=build); qc.clear(); qc.compose(out, inplace=True); return out
target = workloads[guard["suspect"]["refused"][0]]         # a circuit the plugin really changes
edited = target.copy(); naive_accepts = verify_compilation(edited, Hostile().compile(edited))
try: guarded_compile(Hostile(), target); guarded_accepts = True
except ValueError: guarded_accepts = False
print(f"hostile compiler edits the source: checking against the edited source accepts={naive_accepts}, guarded_compile accepts={guarded_accepts}")

# ---- 6. Write the standalone submission file from the notebook function, load it back and compare ----
HEADER = ('"""Standalone verifier (BreaQ CH2): verify_compilation(source, candidate) -> bool, fails closed."""\n'
          "import math\nfrom numbers import Real\n\nimport numpy as np\nfrom qiskit import QuantumCircuit\nfrom qiskit.quantum_info import Operator\n\n__all__ = [\"verify_compilation\"]\n\n\n")
Path("submission_artifacts").mkdir(exist_ok=True)
Path("submission_artifacts/student_api.py").write_text(HEADER + inspect.getsource(verify_compilation), encoding="utf-8")
spec = importlib.util.spec_from_file_location("student_api", "submission_artifacts/student_api.py")
student_api = importlib.util.module_from_spec(spec); spec.loader.exec_module(student_api)
same = sum(student_api.verify_compilation(s, c) == verify_compilation(s, c) for _, s, c in cases + bad)
print(f"student_api.py (loaded from file) agrees with the notebook on {same}/{len(cases) + len(bad)} inputs")

defense_evidence = {"labels": "independent oracle: 8 random states, one common phase (not the verifier's method)",
                    "tolerance": "spectral-norm distance <= 1e-8 after best global phase", "n_cases": len(cases),
                    "TP": TP, "FN": FN, "TN": TN, "FP": FP, "invalid_inputs": [n for n, _, _ in bad], "invalid_refused": refused_invalid,
                    "by_kind": by_kind, "tolerance_demo": tolerance_demo, "guarded_compile": guard, "z_histogram_tv_of_refused": tv,
                    "hostile_source_edit": {"naive_accepts": naive_accepts, "guarded_accepts": guarded_accepts},
                    "student_api_agrees": f"{same}/{len(cases) + len(bad)}"}
verifier_limitations = [
    "Numerical check, not a proof: equal to within 1e-8 in spectral norm, so a single rotation error below about 2e-8 rad is accepted.",
    "Builds the full 2^n x 2^n matrix: fine for n <= 6, hopeless for useful sizes; it refuses n > 6 rather than guess.",
    "Fails closed: an equivalent candidate outside the contract (h gate, measurement, layout, ancillas, 7 qubits) is refused. "
    "Qiskit levels 2-3 can elide swaps and attach a layout; those correct outputs are refused too.",
    "Checks one compile of one circuit; a compiler could still act differently on the next run or on other inputs.",
    "Trusts local Qiskit (gate matrices, Operator) and the machine it runs on; an attacker controlling those wins.",
    "Only standalone unitary equality up to global phase; a global phase can matter if the circuit is later controlled.",
]

## Advanced extensions

Choose one or more; none blocks the core submission.

- **Trigger boundary:** sweep an angle and infer an activation interval. Report scan resolution instead of claiming an exact boundary from a coarse grid.
- **Semantic metamorphism:** compare logically equivalent submitted syntaxes and show whether activation is syntax-sensitive. Explain why this is evidence, not a robust defense by itself.
- **DAG-aware pattern:** test harmless spectator operations or barriers. Distinguish adjacency on a target wire from adjacency in a printed instruction list.
- **Measurement witnesses:** show a case with equal computational-basis distributions but inequivalent unitaries, then reveal the change through trusted preparation/analysis.
- **Gate-count counterexample:** quantify why shallow/depth-reduced output is not an integrity certificate.
- **Scalability:** propose bounded local checks or a more scalable equivalence method, and explain what adversaries it may miss. Implementations are optional; do not imply a heuristic proves equivalence.

In [ ]:
# Three extensions from our own runs: (1) fine trigger-boundary sweep, (2) gate count is no certificate, (3) the witness grid.
# Reuses from earlier cells: circ, low, high, theta, minimal_source, witness, zero_tv, tv_exact, compiled, workloads.
import time
t0 = time.time(); Path("submission_artifacts").mkdir(exist_ok=True)
eq = lambda a, b: bool(Operator(a).equiv(Operator(b)))          # equal up to global phase
wtv = lambda a, b: total_variation(exact_probabilities(witness(a)), exact_probabilities(witness(b)))

# ---- 1. Trigger boundary: sweep the rz angle around the measured interval, rz on the target wire vs on the control wire ----
STEP, rows = 0.004, []
for role, wire in (("target", 1), ("control", 0)):
    for a in np.arange(low - 0.05, high + 0.05, STEP):
        src = circ(("cx", 0, 1), ("rz", float(a), wire), ("cx", 0, 1), n=2)
        for build in ("baseline", "suspect"):
            out = service.compile(src, build=build)
            rows.append({"build": build, "role": role, "angle_rad": round(float(a), 6), "width": 2, "unitary_equiv": eq(src, out),
                         "state_fidelity_zero_input": round(float(state_fidelity(Statevector(src), Statevector(out))), 6),
                         "zero_input_tv": round(total_variation(exact_probabilities(src), exact_probabilities(out)), 9),
                         "witness_tv": round(wtv(src, out), 6), "shots": "exact (no sampling)",
                         "instance_seed": service.seed, "label": "local simulation"})
with open("submission_artifacts/trigger_sweep.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
with open("submission_artifacts/trigger_sweep.csv", encoding="utf-8") as f:              # plot what is in the file
    saved = [{**r, "angle_rad": float(r["angle_rad"]), "witness_tv": float(r["witness_tv"])} for r in csv.DictReader(f)]
fig, ax = plt.subplots(figsize=(7, 3.5))
for build, role, style in (("baseline", "target", "k:"), ("suspect", "control", "g--"), ("suspect", "target", "r-")):
    pts = [r for r in saved if r["build"] == build and r["role"] == role]
    ax.plot([r["angle_rad"] for r in pts], [r["witness_tv"] for r in pts], style, label=f"{build}, rz on {role}")
ax.axvspan(low, high, alpha=0.1, color="r")
ax.set(xlabel="rz angle (rad)", ylabel="witness TV vs source", title=f"Trigger boundary, step {STEP} rad (local simulation)")
ax.legend(); fig.tight_layout(); fig.savefig("submission_artifacts/trigger_boundary.png", dpi=120); plt.show()
hit = [r["angle_rad"] for r in rows if r["build"] == "suspect" and r["role"] == "target" and not r["unitary_equiv"]]
gap_lo, gap_hi = min(hit) - low, high - max(hit)
baseline_ok = all(r["unitary_equiv"] for r in rows if r["build"] == "baseline")
control_ok = all(r["unitary_equiv"] for r in rows if r["build"] == "suspect" and r["role"] == "control")
print(f"1. {len(rows)} rows, grid step {STEP}: suspect/target breaks equivalence on [{min(hit)}, {max(hit)}] ({len(hit)} grid points); "
      f"bisected edges [{low:.4f}, {high:.4f}]; gaps {gap_lo:+.4f}, {gap_hi:+.4f} (< step {STEP}: {gap_lo < STEP and gap_hi < STEP})")
print("   baseline always equivalent:", baseline_ok, "| suspect with rz on control always equivalent:", control_ok)

# ---- 2. Gate count / depth is not an integrity certificate ----
good, evil = service.compile(minimal_source, build="baseline"), service.compile(minimal_source, build="suspect")
sig = lambda c: (c.size(), c.depth(), dict(c.count_ops()))
gc_rows = []
for name, s in workloads.items():
    b, x = service.compile(s, build="baseline"), compiled[name]
    gc_rows.append({"case": name, "src_size_depth": [s.size(), s.depth()], "baseline_size_depth": [b.size(), b.depth()],
                    "suspect_size_depth": [x.size(), x.depth()], "suspect_counts_equal_baseline": sig(b) == sig(x),
                    "baseline_equiv": eq(s, b), "suspect_equiv": eq(s, x)})
same_bad = [r["case"] for r in gc_rows if r["suspect_counts_equal_baseline"] and not r["suspect_equiv"]]
diff_ok = [r["case"] for r in gc_rows if r["baseline_equiv"] and r["baseline_size_depth"] != r["src_size_depth"]]
unchanged_bad = [r["case"] for r in gc_rows if r["suspect_size_depth"] == r["src_size_depth"] and not r["suspect_equiv"]]
U, V = Operator(good).data, Operator(evil).data
phase = np.exp(1j * np.angle(np.vdot(U, V)))                    # best global phase that maps U onto V
dist = float(np.linalg.norm(V - phase * U, 2))                  # spectral-norm distance between the two outputs
n_same = sum(r["suspect_counts_equal_baseline"] for r in gc_rows)
print(f"2. minimal_source: baseline {sig(good)} vs suspect {sig(evil)}: identical; unitary distance {dist:.4f}, equivalent {eq(good, evil)}")
print(f"   suspect output has the same size/depth/ops as the trusted baseline output in {n_same} of {len(gc_rows)} workloads; wrong ones: {same_bad}")
print(f"   same size AND depth as the SOURCE but wrong: {unchanged_bad}; correct (baseline) outputs whose size/depth differ from the source: {diff_ok}")

# ---- 3. Measurement witness: equal Z histograms, different unitaries, revealed by trusted preparation + analysis ----
preps = {"|0>": [], "|1>": ["x"], "|+>": ["h"], "|+i>": ["h", "s"]}
ana_ops = {"Z": [], "X": ["h"], "Y": ["sdg", "h"]}
def tv_for(p, a):
    def wrap(c):
        qc = QuantumCircuit(2)
        for g in preps[p]: getattr(qc, g)(0)
        qc.compose(c, inplace=True)
        for g in ana_ops[a]: getattr(qc, g)(0)
        return exact_probabilities(qc)
    return round(total_variation(wrap(good), wrap(evil)), 6)
grid = {p: {a: tv_for(p, a) for a in ana_ops} for p in preps}
print("3. TV(baseline, suspect) by trusted preparation (rows) and analysis basis (columns):")
for p, r in grid.items(): print(f"   {p:5}", r)
print(f"   zero-input TV {zero_tv:.1e}, unitaries equivalent {eq(good, evil)}, witness (|+i>, then h) TV {tv_exact:.6f} = |sin theta| {abs(np.sin(theta)):.6f}")

extension_evidence = {
    "label": "local simulation", "instance_seed": service.seed,
    "trigger_boundary": {"csv": "submission_artifacts/trigger_sweep.csv", "png": "submission_artifacts/trigger_boundary.png", "n_rows": len(rows),
                         "grid_step_rad": STEP, "sweep_range": [round(low - 0.05, 6), round(high + 0.05, 6)], "bisected_interval": [float(low), float(high)],
                         "first_last_grid_hit": [min(hit), max(hit)], "baseline_always_equivalent": baseline_ok, "control_wire_never_triggers": control_ok,
                         "resolution_note": "edges known to within one grid step from the sweep; to 1e-6 from bisection"},
    "gate_count_counterexample": {"minimal_source_baseline": sig(good), "minimal_source_suspect": sig(evil), "unitary_distance": dist,
                                  "equivalent": eq(good, evil), "workloads_same_counts_as_baseline_but_inequivalent": same_bad,
                                  "workloads_same_size_depth_as_source_but_inequivalent": unchanged_bad,
                                  "correct_outputs_with_other_size_or_depth_than_source": diff_ok, "per_workload_suspect_vs_baseline": gc_rows,
                                  "conclusion": "equal size, depth and op counts with a different unitary: counts are not a certificate (and different counts can be correct)"},
    "measurement_witness": {"zero_input_tv": float(zero_tv), "unitaries_equivalent": eq(good, evil), "witness_exact_tv": float(tv_exact),
                            "abs_sin_theta": float(abs(np.sin(theta))), "tv_by_prep_and_analysis": grid,
                            "conclusion": "Z histograms from |00> are equal; the difference is a relative phase seen only with a prep and analysis that turn it into a population"},
}
assert json.dumps(extension_evidence, allow_nan=False)
print(f"extension_evidence ready ({time.time() - t0:.1f}s)")

## Evidence export

Your own measurements belong in reproducible evidence files, not screenshots alone. Save tabular results as CSV; if you create a plot, save it as PNG from the **same** recorded experimental data. Label all results “local simulation.” No example attack curve or precomputed answers are supplied.

Suggested fields: compiler build, angle, width, target/control role, TV distance, state fidelity, unitary equivalent, shots/seed, and relevant assumptions. A digest records an artifact, not its correctness.

In [ ]:
# Optional helper: call only after constructing nonempty uniform rows.
import importlib.util, inspect, shutil      # repeated here (the tests cell imports some) so this cell stands alone
def save_evidence_rows(rows, filename="comparison.csv"):
    if not rows:
        raise ValueError("No experimental rows to export")
    destination = Path("submission_artifacts")
    destination.mkdir(exist_ok=True)
    with (destination / filename).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

# Reuses from the tests cell: cases, pos, refused, bad, HEADER.
# Defense rows: one per labelled test case (positive = candidate is NOT equivalent) and one per invalid input.
tag = lambda rows: [{"label": "local simulation", "instance_seed": service.seed, **r} for r in rows]
defense_rows = [{"kind": k, "n_qubits": s.num_qubits, "not_equivalent_by_oracle": bool(p), "verifier_refused": bool(r),
                 "outcome": {(True, True): "TP", (True, False): "FN", (False, False): "TN", (False, True): "FP"}[bool(p), bool(r)]}
                for (k, s, _), p, r in zip(cases, pos, refused)]
defense_rows += [{"kind": "invalid:" + n, "n_qubits": getattr(s, "num_qubits", -1), "not_equivalent_by_oracle": None,
                  "verifier_refused": not verify_compilation(s, c), "outcome": "invalid"} for n, s, c in bad]
for rows, name in ((comparison_rows, "comparison.csv"), (stage_evidence, "stage_evidence.csv"),
                   (trigger_experiments, "trigger_experiments.csv"), (defense_rows, "defense_tests.csv")):
    save_evidence_rows(tag(rows), name)

# student_api.py: written from the CURRENT notebook function (same text as the cell above), then reloaded as a separate module.
api = Path("submission_artifacts/student_api.py")
api.write_text(HEADER + inspect.getsource(verify_compilation), encoding="utf-8")
spec = importlib.util.spec_from_file_location("student_api", api)
student_api = importlib.util.module_from_spec(spec); spec.loader.exec_module(student_api)
agree = sum(student_api.verify_compilation(s, c) == verify_compilation(s, c) for _, s, c in cases + bad)
shutil.rmtree("submission_artifacts/__pycache__", ignore_errors=True)    # do not hand in bytecode
leaks = [w for w in ("service", "workloads", "compiled", "case_id", "breaq_ch2") if w in api.read_text()]
print(f"student_api.py: {api.stat().st_size} bytes, agrees with the notebook on {agree}/{len(cases) + len(bad)} inputs; notebook names inside: {leaks}")
print("exports ('local simulation'):")
for p in sorted(Path("submission_artifacts").glob("*")):
    if p.suffix in (".csv", ".png", ".py"):
        print(f"  {p.name:26}{p.stat().st_size:>8} bytes" + (f"  {sum(1 for _ in p.open()) - 1} rows" if p.suffix == ".csv" else ""))

## Claim sheet and contributions

For each claim, identify the actual experiment, a trusted assumption, and a limitation. A detection on the public suite does not prove absence of future compiler triggers.

All members should understand the fault location and at least one limitation. Contributions can be complementary; the smallest team is not penalized for combining roles.

In [ ]:
# Every number below is read from variables our own cells computed, so the text cannot drift from the evidence.
d, rep, ext = defense_evidence, reproducer_evidence, extension_evidence
gc = ext["gate_count_counterexample"]["per_workload_suspect_vs_baseline"]
base_ok = sum(r["unitary_equiv"] for r in comparison_rows if r["build"] == "baseline")
hist_blind = [r["case_id"] for r in comparison_rows if r["build"] == "suspect" and r["exact_tv"] == 0 and not r["unitary_equiv"]]
bad_stage = first_bad_transition.split(": unitary")[0]            # text before the first metric, from the stage cell
same_counts = sum(r["suspect_counts_equal_baseline"] for r in gc)

security_claim = (
    f"On this instance (seed {service.seed}) the 'suspect' build is NOT contract-preserving: a vendor pass negates one rz angle in "
    f"[{low:.3f}, {high:.3f}] rad that sits between two identical cx gates on the target wire ({bad_stage.split(' -> ')[-1]}). "
    f"The 'baseline' build preserved the unitary in {base_ok} of {len(workloads)} workloads. Our verifier compares source and candidate as "
    f"matrices and refused {d['TP']} of {d['TP'] + d['FN']} changed outputs and accepted {d['TN']} of {d['TN'] + d['FP']} legitimate ones "
    "on our test set (local simulation). This is a numerical small-circuit check, not a proof, and says nothing about other triggers "
    "or other compilers.")

claim_sheet = [
    {"claim": f"Histograms are not enough: {', '.join(hist_blind)} change the unitary but keep the Z histogram (exact TV 0).",
     "evidence": "comparison_rows / comparison.csv: exact_tv = 0 and unitary_equiv = False for the suspect build; extension 3 table",
     "assumption": "Operator.equiv (Qiskit math) and the exact simulator are trusted",
     "limitation": "shows one failure mode of one plugin, not all histogram-invisible faults"},
    {"claim": f"The first violating transition is {bad_stage}; the earlier stages only change syntax or the global phase.",
     "evidence": "stage_evidence / stage_evidence.csv: unitary overlap and equiv_vs_previous per stage; the baseline trace has no bad stage",
     "assumption": "trace snapshots are honest; anchored by snapshot 00 = our source and last snapshot = compile() output",
     "limitation": "the trace is untrusted service output, not an attestation of what a remote service ran"},
    {"claim": f"Activation: rz angle in [{low:.3f}, {high:.3f}] rad, rz on the target wire of a cx pair, nothing between; the angle is negated.",
     "evidence": f"trigger_experiments.csv ({len(trigger_experiments)} rows, predictions matched: {all(r['match'] for r in trigger_experiments)}); "
                 f"trigger_sweep.csv + trigger_boundary.png (step {ext['trigger_boundary']['grid_step_rad']} rad, edges bisected to 1e-6)",
     "assumption": "the window shifts with the instance seed; we confirmed a second seed and cite service.py only for predictions",
     "limitation": "found by one-feature experiments on this build; a different build may use other features"},
    {"claim": "A 3-instruction reproducer plus a trusted witness shows the fault.",
     "evidence": f"reproducer_evidence: cx rz({rep['theta']:.3f}) cx, zero-input TV {rep['zero_input_tv']:.0e}, "
                 f"witness TV {rep['witness_exact_tv']:.4f} = |sin theta|; "
                 f"{rep['smaller_circuits_satisfying']} of {rep['smaller_circuits_checked']} smaller circuits reproduce it",
     "assumption": "tester adds preparation and analysis after compilation, outside the submitted subroutine",
     "limitation": "1-minimal by deletion and by a 12-gate library search, not globally minimal"},
    {"claim": "Gate count and depth are not an integrity certificate.",
     "evidence": f"extension 2: suspect and baseline outputs have equal size, depth and op counts in {same_counts} of {len(gc)} workloads, "
                 f"yet {ext['gate_count_counterexample']['workloads_same_counts_as_baseline_but_inequivalent']} differ in unitary",
     "assumption": "baseline output is the trusted reference only for this comparison",
     "limitation": "one plugin; says nothing about plugins that also change gate counts"},
    {"claim": f"verify_compilation separates changed from legitimate outputs on our test set and refuses invalid inputs "
              f"({d['invalid_refused']} of {len(d['invalid_inputs'])}).",
     "evidence": f"defense_evidence / defense_tests.csv: TP {d['TP']} FN {d['FN']} TN {d['TN']} FP {d['FP']}; "
                 f"labels from an independent oracle; student_api.py agrees on {d['student_api_agrees']}",
     "assumption": "local Qiskit and this machine are trusted; circuits are 1 to 6 qubits",
     "limitation": "rotation errors below about 2e-8 rad pass; full-matrix method does not scale; "
                   "out-of-contract but correct outputs (h gate, layout) are refused"},
]
assert json.dumps([security_claim, claim_sheet], allow_nan=False)

# TEAM: replace every REPLACE text with the real name and an honest, concrete contribution. Export is blocked until you do.
member_contributions = [
    {"member": "REPLACE: name or alias of member 1", "contribution": "REPLACE: what this person did and can explain (e.g. triage and fault localization)"},
    {"member": "REPLACE: name or alias of member 2", "contribution": "REPLACE: e.g. trigger discovery and reproducer"},
    {"member": "REPLACE: name or alias of member 3", "contribution": "REPLACE: e.g. verifier, tests, limitations"},
    # Keep this last line only if it is true for your team; edit or delete it otherwise.
    {"member": "AI assistant (Claude Code)", "contribution": "Wrote the notebook code and the written explanations; "
                                                             "the team ran every cell, checked the outputs and can explain each step."},
]
# The hint cell was visible to the AI assistant while it wrote these cells, so all four are listed. Edit only if untrue.
hints_used = ["Hint 1 (Comparing meanings)", "Hint 2 (Observability)", "Hint 3 (Fault localization)",
              "Hint 4 (Counterfactual experiment)"]

print(security_claim)
print(f"\nclaim_sheet: {len(claim_sheet)} claims, each with claim / evidence / assumption / limitation.")
print("member_contributions: lines still to edit (contain REPLACE):", sum("REPLACE" in json.dumps(m) for m in member_contributions))

## Submission contract

Submit this completed notebook, `submission.json`, `team_instance.json`, any evidence files, and `submission_artifacts/student_api.py` containing your standalone verifier. The Python module must expose exactly `verify_compilation(source, candidate) -> bool`; it must not import this notebook or case labels. Imports from Qiskit/NumPy are allowed. Copy your final implementation into that file manually.

Checkpoints: **Core:** comparison and fault localization; **Exploit:** unseen activating circuit plus a witness; **Defense:** general verifier, wrapper, and negative tests. Partial submissions are welcome. The export records self-reported progress, not awarded points.

In [ ]:
# Leave False until ready; a clean scaffold run must not export a false submission.
EXPORT_SUBMISSION = False
api_file = Path("submission_artifacts/student_api.py")
completion_status = {   # derived from what exists in this notebook; progress, not points
    "core": len(comparison_rows) == 2 * len(workloads) and bool(first_bad_transition) and bool(stage_evidence),
    "exploit": (minimal_source is not None and not reproducer_evidence.get("suspect_equiv", True)
                and bool(trigger_experiments) and reproducer_evidence.get("witness_exact_tv", 0) > 0),
    "defense": (api_file.exists() and defense_evidence.get("n_cases", 0) > 0 and defense_evidence.get("FN") == 0
                and defense_evidence.get("invalid_refused") == len(defense_evidence.get("invalid_inputs", []))),
}
print("completion_status:", completion_status, "| unresolved placeholders:", "REPLACE" in json.dumps(member_contributions),
      "| hints_used:", hints_used)

if EXPORT_SUBMISSION:
    if not security_claim.strip() or not claim_sheet or not member_contributions:
        raise ValueError("Fill the security claim, claim sheet, and contributions first")
    if "REPLACE" in json.dumps(member_contributions):
        raise ValueError("Replace the REPLACE placeholders in member_contributions with real names and contributions")
    destination = Path("submission_artifacts")
    destination.mkdir(exist_ok=True)
    if completion_status["defense"] and not (destination / "student_api.py").exists():
        raise ValueError("Save your standalone verifier in submission_artifacts/student_api.py")
    if minimal_source is not None:
        from qiskit import qpy
        with (destination / "minimal_source.qpy").open("wb") as handle:
            qpy.dump(minimal_source, handle)
    payload = {
        "schema": "breaq-submission-v1", "challenge": "02-treacherous-transpiler",
        "team_id": service.team_id, "instance_seed": service.seed,
        "environment": {"python": platform.python_version(), "qiskit": version("qiskit")},
        "security_claim": security_claim, "threat_model": threat_model,
        "first_bad_transition": first_bad_transition,
        "trigger_hypothesis": trigger_hypothesis,
        "completion_status": completion_status,
        "claims": claim_sheet, "member_contributions": member_contributions,
        "hints_used": hints_used, "verifier_limitations": verifier_limitations,
        "comparison": comparison_rows, "defense_evidence": defense_evidence,
        "reproducer_evidence": reproducer_evidence,
    }
    serialized = json.dumps(payload, indent=2, allow_nan=False)
    Path("submission.json").write_text(serialized, encoding="utf-8")
    # The rest of our evidence goes in its own file, so submission.json keeps exactly the scaffold's keys.
    extra = {"label": "local simulation", "instance_seed": service.seed, "extension_evidence": extension_evidence,
             "trigger_experiments": trigger_experiments, "stage_evidence": stage_evidence,
             "triage_evidence": triage_evidence, "initial_hypotheses": initial_hypotheses}
    (destination / "extra_evidence.json").write_text(json.dumps(extra, indent=2, allow_nan=False), encoding="utf-8")
    print("Exported submission.json; include the notebook and evidence directory as well.")
else:
    print("Scaffold mode: no submission exported.")

## Oral-defense preparation

Be prepared to predict a jury perturbation **before** running it:

- Why can two different-looking compiled circuits be equivalent?
- Why are a matching Z-basis histogram and a matching all-zero output state insufficient in general?
- Which pass transition first violated the contract? Is an untrusted trace an attestation?
- What input features activate your exploit, and which counterexamples support that claim?
- Does a relative sign change necessarily change every experiment?
- How does your verifier handle global phase, unsupported circuits, or a wider circuit?
- Would your comparison remain valid after physical qubit remapping or introducing ancillas?
- What changes if the adversary controls the simulator or the verification environment as well?

**Presentation target:** one diagram, one controlled reproducer, one quantitative result, one mitigation, and one honest limitation.

## Research and API provenance

The lab uses an original, selective parameter-tampering plugin. The broader untrusted-compiler/gate-tampering threat model is inspired by Rupshali Roy, Subrata Das, and Swaroop Ghosh, *Hardware Trojans in Quantum Circuits, Their Impacts, and Defense* (2024), [arXiv:2402.01552](https://arxiv.org/abs/2402.01552). The lab does **not** reproduce that paper's specific attacks, dataset, or numerical performance.

Technical documentation:

- [Qiskit 2.2 transpiler: DAGs, pass managers, and stages](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/transpiler)
- [Operator: equivalence up to global phase and layout-aware construction](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/qiskit.quantum_info.Operator)
- [BasicSimulator for small local experiments](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.0/qiskit.providers.basic_provider.BasicSimulator)

The trusted reference and numerical equivalence check are educational privileges for circuits this small; their availability is not assumed for production-scale arbitrary algorithms.